# Train YOLOv8-seg cho bãi xe mô hình (DIP)

Chạy **lần lượt từ trên xuống**, mỗi cell bấm ▶ và đợi xong mới chạy cell tiếp.
Trước khi chạy: **Runtime → Change runtime type → T4 GPU → Save**.

Chỉ có **1 chỗ phải sửa**: Cell 3 (dán đoạn download code từ Roboflow).

## Cell 1. Kiểm tra GPU và nối Google Drive (để không mất kết quả nếu Colab tự ngắt)

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/dip_baixe'   # kết quả train sẽ lưu ở đây
import os; os.makedirs(OUT, exist_ok=True)

## Cell 2. Cài thư viện

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics; ultralytics.checks()

## Cell 3. Tải dataset từ Roboflow  ✏️ SỬA CELL NÀY

Roboflow → Versions → chọn version vừa Generate → **Download Dataset** → Format **YOLOv8** → **Show download code** → copy.
Xóa 5 dòng mẫu bên dưới rồi **dán đè đoạn code đó vào** (giữ lại 2 dòng `%cd` ở đầu).

In [ ]:
%cd /content
# ==== DÁN ĐOẠN DOWNLOAD CODE CỦA ROBOFLOW VÀO ĐÂY (thay 5 dòng dưới) ====
from roboflow import Roboflow
rf = Roboflow(api_key="DAN_API_KEY_O_DAY")
project = rf.workspace("TEN_WORKSPACE").project("baixe-dip")
version = project.version(1)
dataset = version.download("yolov8")
# ======================================================================

## Cell 4. Kiểm tra số ảnh (phải ra khoảng train 207, valid 15, test 22)

In [ ]:
import glob
DATA = dataset.location
for s in ['train', 'valid', 'test']:
    n_img = len(glob.glob(f'{DATA}/{s}/images/*'))
    n_lbl = len(glob.glob(f'{DATA}/{s}/labels/*.txt'))
    print(f'{s:5s}: {n_img} ảnh, {n_lbl} file nhãn')
print(open(f'{DATA}/data.yaml').read())

## Cell 4b. Dọn nhãn hộp (bbox) lẫn trong dataset, chạy trước khi train

In [ ]:
import os, glob
# Tìm nhãn chỉ là khung chữ nhật (5 số) thay vì polygon, rồi bỏ cả ảnh đó ra khỏi dataset
bad = []
for s in ['train', 'valid', 'test']:
    for lbl in glob.glob(f'{DATA}/{s}/labels/*.txt'):
        lines = [l.split() for l in open(lbl) if l.strip()]
        if any(len(l) < 7 for l in lines):
            bad.append(lbl)
for lbl in bad:
    print('Nhãn hộp, bỏ ảnh:', lbl.split('/labels/')[0].split('/')[-1], os.path.basename(lbl))
    stem = os.path.splitext(os.path.basename(lbl))[0]
    for img in glob.glob(lbl.replace('/labels/', '/images/').rsplit('.', 1)[0] + '.*'):
        os.remove(img)
    os.remove(lbl)
for c in glob.glob(f'{DATA}/**/*.cache', recursive=True):
    os.remove(c)   # xóa cache nhãn cũ của YOLO
print('Đã bỏ', len(bad), 'ảnh. Giờ chạy Cell 5.')

## Cell 5. Train (lâu nhất, để nguyên tab Colab mở)

Nếu báo `CUDA out of memory`: đổi `batch=8` thành `batch=4` rồi chạy lại cell này.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s-seg.pt')
model.train(data=f'{DATA}/data.yaml', imgsz=1024, epochs=150, batch=8, patience=40,
            project=OUT, name='train', exist_ok=True)

## Cell 6. Xem biểu đồ train và ảnh dự đoán trên tập valid

In [ ]:
from IPython.display import Image, display
RUN = f'{OUT}/train'
for f in ['results.png', 'MaskPR_curve.png', 'val_batch0_pred.jpg']:
    if os.path.exists(f'{RUN}/{f}'):
        print(f); display(Image(filename=f'{RUN}/{f}', width=900))

## Cell 7. Đánh giá trên bộ TEST (ảnh mô hình chưa thấy bao giờ), lấy số cho báo cáo

In [ ]:
best = YOLO(f'{RUN}/weights/best.pt')
m = best.val(data=f'{DATA}/data.yaml', split='test', imgsz=1024, project=OUT, name='test_eval', exist_ok=True)
print(f'Box  mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')
print(f'Mask mAP50 = {m.seg.map50:.3f}   mAP50-95 = {m.seg.map:.3f}')

## Cell 8. Vẽ viền xe lên ảnh TEST (retina_masks = mask ở độ phân giải gốc)

In [ ]:
res = best.predict(source=f'{DATA}/test/images', imgsz=1024, conf=0.5, retina_masks=True,
                   save=True, project=OUT, name='test_pred', exist_ok=True)
for r in res[:4]:
    print(os.path.basename(r.path), '->', len(r.boxes), 'xe')
    display(Image(filename=f'{OUT}/test_pred/{os.path.basename(r.path)}', width=900))

## Xong

Mọi thứ đã nằm trong Google Drive: **MyDrive/dip_baixe/**
- `train/weights/best.pt`: mô hình để dùng tiếp
- `train/results.png`, `test_eval/`, `test_pred/`: hình cho báo cáo